# EDA do Datathon

Este notebook documenta a análise exploratória mínima usada para entender as bases `PEDE2022`, `PEDE2023` e `PEDE2024`, validar diferenças de schema e apoiar as decisões do pipeline temporal de ML.

Escopo:
- comparar estrutura entre anos;
- medir diferenças de schema;
- contar tokens inválidos relevantes;
- medir interseção de `RA` entre anos;
- reforçar a definição do target temporal;
- registrar a limitação semântica de `Idade x Fase Ideal`.


In [1]:
import os
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'dataset').exists() and (repo_root.parent / 'dataset').exists():
    repo_root = repo_root.parent

dataset_path = Path(
    os.environ.get('DATASET_PATH', repo_root / 'dataset' / 'DATATHON' / 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx')
)
dataset_path


PosixPath('/Users/njunior/workspace/repos/fiap/fiap-techchalenge-f5/dataset/DATATHON/BASE DE DADOS PEDE 2024 - DATATHON.xlsx')

## Carregamento das abas principais

Usamos dois carregamentos: um padrão para exploração geral e um preservando strings brutas para contar tokens inválidos originados da planilha.


In [2]:
sheet_names = ['PEDE2022', 'PEDE2023', 'PEDE2024']
dfs = {name: pd.read_excel(dataset_path, sheet_name=name) for name in sheet_names}
raw_dfs = {name: pd.read_excel(dataset_path, sheet_name=name, keep_default_na=False) for name in sheet_names}
{name: df.shape for name, df in dfs.items()}


{'PEDE2022': (860, 42), 'PEDE2023': (1014, 48), 'PEDE2024': (1156, 50)}

In [3]:
summary = pd.DataFrame(
    [
        {
            'sheet': name,
            'rows': df.shape[0],
            'cols': df.shape[1],
            'has_RA': 'RA' in df.columns,
            'defas_col': 'Defasagem' if 'Defasagem' in df.columns else ('Defas' if 'Defas' in df.columns else None),
        }
        for name, df in dfs.items()
    ]
)
summary


      sheet  rows  cols  has_RA  defas_col
0  PEDE2022   860    42    True      Defas
1  PEDE2023  1014    48    True  Defasagem
2  PEDE2024  1156    50    True  Defasagem

## Diferenças de schema entre anos

A ingestão do projeto não é direta: há mudança de nomes de colunas, colunas extras por ano e variações de tipagem.


In [4]:
pairs = [('PEDE2022', 'PEDE2023'), ('PEDE2023', 'PEDE2024'), ('PEDE2022', 'PEDE2024')]
schema_compare = []
for left_name, right_name in pairs:
    left_cols = set(dfs[left_name].columns)
    right_cols = set(dfs[right_name].columns)
    schema_compare.append({
        'pair': f'{left_name} -> {right_name}',
        'shared_cols': len(left_cols & right_cols),
        'only_left': len(left_cols - right_cols),
        'only_right': len(right_cols - left_cols),
    })
pd.DataFrame(schema_compare)


                   pair  shared_cols  only_left  only_right
0  PEDE2022 -> PEDE2023           34          8          14
1  PEDE2023 -> PEDE2024           43          5           7
2  PEDE2022 -> PEDE2024           32         10          18

In [5]:
pd.DataFrame({
    'only_2022_vs_2023': pd.Series(sorted(set(dfs['PEDE2022'].columns) - set(dfs['PEDE2023'].columns))[:10]),
    'only_2023_vs_2022': pd.Series(sorted(set(dfs['PEDE2023'].columns) - set(dfs['PEDE2022'].columns))[:10]),
})


  only_2022_vs_2023 only_2023_vs_2022
0          Ano nasc      Data de Nasc
1             Defas         Defasagem
2        Fase ideal    Destaque IPV.1
3          Idade 22        Fase Ideal
4            Inglês         INDE 2023
5             Matem           INDE 23
6              Nome               IPP
7            Portug             Idade
8               NaN               Ing
9               NaN               Mat

## Qualidade básica: tokens inválidos de planilha\n\nPara essa contagem, usamos leitura bruta com `openpyxl`, porque `pandas` normaliza parte dos erros de planilha para `NaN` e esconderia parte do problema operacional.\n

In [6]:
from openpyxl import load_workbook

invalid_tokens = ['#N/A', '#DIV/0!', 'INCLUIR']
wb = load_workbook(dataset_path, data_only=False, read_only=True)
quality_rows = []
for name in sheet_names:
    ws = wb[name]
    counts = {token: 0 for token in invalid_tokens}
    for row in ws.iter_rows(values_only=True):
        for value in row:
            if value in counts:
                counts[value] += 1
    counts['sheet'] = name
    counts['total_invalid_tokens'] = sum(counts[token] for token in invalid_tokens)
    quality_rows.append(counts)
pd.DataFrame(quality_rows)[['sheet', '#N/A', '#DIV/0!', 'INCLUIR', 'total_invalid_tokens']]


      sheet  #N/A  #DIV/0!  INCLUIR  total_invalid_tokens
0  PEDE2022     0        0        0                     0
1  PEDE2023   385        3        0                   388
2  PEDE2024   458      227       76                   761

## Coorte temporal por RA

O pipeline oficial usa pares temporais por `RA`, então o tamanho da interseção entre anos é um dado estrutural importante.


In [7]:
ra_sets = {name: set(dfs[name]['RA'].dropna().astype(str)) for name in sheet_names}
cohort_rows = []
for left_name, right_name in pairs:
    inter = ra_sets[left_name] & ra_sets[right_name]
    cohort_rows.append({
        'pair': f'{left_name} ∩ {right_name}',
        'intersection_ra': len(inter),
        f'%_{left_name}': round(100 * len(inter) / max(len(ra_sets[left_name]), 1), 1),
        f'%_{right_name}': round(100 * len(inter) / max(len(ra_sets[right_name]), 1), 1),
    })
pd.DataFrame(cohort_rows)


                  pair  intersection_ra  %_PEDE2022  %_PEDE2023  %_PEDE2024
0  PEDE2022 ∩ PEDE2023              600        69.8        59.2         NaN
1  PEDE2023 ∩ PEDE2024              765         NaN        75.4        66.2
2  PEDE2022 ∩ PEDE2024              472        54.9         NaN        40.8

## Definição do target temporal

No projeto, a predição é feita no ano `t` para estimar a defasagem em `t+1`.

Definição oficial:
- `y = 1` quando `Defasagem_{t+1} < 0`;
- `y = 0` caso contrário.

Recortes usados:
- treino: `X(2022) -> y(2023)`
- holdout: `X(2023) -> y(2024)`


In [8]:
target_preview = dfs['PEDE2024'][['RA', 'Defasagem']].copy()
target_preview['target_if_year_t1'] = (pd.to_numeric(target_preview['Defasagem'], errors='coerce') < 0).astype('Int64')
target_preview.head()


        RA  Defasagem  target_if_year_t1
0  RA-1275          0                  0
1  RA-1276          0                  0
2  RA-1277          0                  0
3   RA-868         -1                  1
4  RA-1278         -1                  1

## Limitação conhecida: `Idade x Fase Ideal`

Há inconsistências semânticas no dataset em que estudantes da mesma idade aparecem com `Fase Ideal` diferente.

Nesta entrega, a coluna foi tratada como categórica observada, sem recalcular a fase ideal a partir da idade.


In [9]:
fase_check = dfs['PEDE2024'][['Idade', 'Fase Ideal']].dropna().copy()
phase_variation = (
    fase_check.groupby('Idade')['Fase Ideal']
    .nunique()
    .reset_index(name='distinct_fase_ideal')
    .query('distinct_fase_ideal > 1')
    .sort_values(['distinct_fase_ideal', 'Idade'], ascending=[False, True])
)
phase_variation.head(10)


    Idade  distinct_fase_ideal
7      14                    3
8      15                    3
1       8                    2
3      10                    2
4      11                    2
5      12                    2
6      13                    2
9      16                    2
10     17                    2
11     18                    2

## Fechamento

Este notebook é um artefato exploratório complementar. As decisões finais de engenharia usadas no projeto estão consolidadas em:
- `docs/analise_bases_e_dicionario.md`;
- `docs/column_mapping.md`;
- `src/data.py`, `src/contracts.py`, `src/validate.py` e pipeline de treino/serving.
